## Trainer realizado en google colab

In [1]:
!pip install -q transformers datasets accelerate scikit-learn pymongo pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 16.6 MB/s eta 0:00:00


In [3]:
from pymongo import MongoClient
import pandas as pd
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
client = MongoClient(MONGO_URI)
db = client["dbx_Canciones"]
coleccion = db["canciones"]

cursor = coleccion.find({})
df = pd.DataFrame(list(cursor))
df.columns = df.columns.str.strip()
df = df[df["letra"].notna()].copy()

print(f"Total: {len(df)}")
print(df.columns.tolist())

Total: 10384
['_id', 'titulo', 'artista', 'genero', 'anio', 'letra', 'idioma', 'fuente', 'url_fuente', 'fecha_recopilacion', 'pos_tags', 'embeddings', 'metricas']


## Etiquetas de décadas

In [4]:
def anio_a_decada(anio):
    try:
        anio = int(anio)
        if 1990 <= anio <= 1999: return "90s"
        if 2000 <= anio <= 2009: return "2000s"
        if 2010 <= anio <= 2019: return "2010s"
        if 2020 <= anio <= 2029: return "2020s"
    except: pass
    return None

df["decada"] = df["anio"].apply(anio_a_decada)
df = df[df["decada"].notna()].copy()
df = df[df["letra"].str.len() > 50].copy()

df_balanceado = df.groupby("decada").apply(
    lambda x: x.sample(n=min(len(x), 900), random_state=42)
).reset_index(drop=True)

print("Distribución balanceada:")
print(df_balanceado["decada"].value_counts().sort_index())
print(f"\nTotal: {len(df_balanceado)}")

Distribución balanceada:
decada
2000s    900
2010s    900
2020s    900
90s      864
Name: count, dtype: int64

Total: 3564


/tmp/ipykernel_676/269466688.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanceado = df.groupby("decada").apply(


## Corpus con Split 70/15/15

In [5]:
"""70% Train - 15% Validation - 15% Test"""
from sklearn.model_selection import train_test_split

decadas = sorted(df_balanceado["decada"].unique())
label2id = {d: i for i, d in enumerate(decadas)}
id2label = {i: d for d, i in label2id.items()}
df_balanceado["label"] = df_balanceado["decada"].map(label2id)

train_df, temp_df = train_test_split(df_balanceado, test_size=0.30, random_state=42, stratify=df_balanceado["label"])
val_df,   test_df = train_test_split(temp_df,        test_size=0.50, random_state=42, stratify=temp_df["label"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Décadas: {label2id}")

Train: 2494 | Val: 535 | Test: 535
Décadas: {'2000s': 0, '2010s': 1, '2020s': 2, '90s': 3}


## Tokenizer

In [6]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenizar(batch):
    return tokenizer(batch["letra"], truncation=True, padding="max_length", max_length=256)

train_ds = Dataset.from_pandas(train_df[["letra", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["letra",  "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["letra", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenizar, batched=True)
val_ds   = val_ds.map(tokenizar,   batched=True)
test_ds  = test_ds.map(tokenizar,  batched=True)

print("Datasets tokenizados.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2494 [00:00<?, ? examples/s]

Map:   0%|          | 0/535 [00:00<?, ? examples/s]

Map:   0%|          | 0/535 [00:00<?, ? examples/s]

Datasets tokenizados.


## Fine tunning con Trainer

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="clasificador_decadas",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.347614,0.332710,0.282459
2,No log,1.319653,0.379439,0.370537
3,No log,1.310183,0.385047,0.384390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=234, training_loss=1.251706767285991, metrics={'train_runtime': 71.2102, 'train_samples_per_second': 105.069, 'train_steps_per_second': 3.286, 'total_flos': 495578213609472.0, 'train_loss': 1.251706767285991, 'epoch': 3.0})

In [8]:
import shutil # generar un zip de los resultados del trainer con fine tunning
from google.colab import files

shutil.make_archive("clasificador_decadas", "zip", "clasificador_decadas")

'/content/clasificador_decadas.zip'

In [ ]:
from google.colab import files # opción para guardarlo directo a local
files.download("clasificador_decadas.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from google.colab import drive # opción subirlo a drive para descargarlo desde ahí
drive.mount('/content/drive', force_remount=True)

import shutil
shutil.copy("clasificador_decadas.zip", "/content/drive/MyDrive/clasificador_decadas.zip")
print("Listo!")

Mounted at /content/drive
Listo!


In [12]:
from transformers import AutoTokenizer # archivos que faltaron
from google.colab import files
import shutil, os

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
os.makedirs("tokenizer_decadas", exist_ok=True)
tokenizer.save_pretrained("tokenizer_decadas")

shutil.make_archive("tokenizer_decadas", "zip", "tokenizer_decadas")
files.download("tokenizer_decadas.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>